# Big Data with Arrow

## What you'll learn
- What Apache Arrow is and why it makes data processing faster
- How to read files quickly with `read_csv_arrow()`
- Arrow Tables vs R data frames
- Using dplyr verbs on Arrow objects
- The Parquet file format — a better alternative to CSV for large data

## Prerequisites
- Completed Notebooks 01 and 02
- Generated the large dataset: `python scripts/generate_large_data.py`

## What Is Apache Arrow?

Apache Arrow is a technology for working with data in a highly efficient **columnar** format. Instead of storing data row by row (like a CSV), it stores data column by column. This may sound like a small difference, but it has big consequences:

- **Faster reads:** When you only need a few columns, Arrow reads just those columns instead of the entire file
- **Less memory:** Arrow's format is compact and avoids unnecessary copies
- **Faster operations:** Calculations on a single column are faster when that column's data is stored contiguously

The `arrow` R package brings these benefits to your R workflow. You can use it as a drop-in upgrade for reading files, and you can use dplyr verbs on Arrow objects just like you would on a regular data frame.

In [ ]:
library(arrow)
library(dplyr)
library(readr)

## Reading CSV Files with Arrow

The simplest way to start using Arrow is to swap `read_csv()` for `read_csv_arrow()`. Let's compare them.

In [ ]:
# Standard readr (tidyverse) read
system.time({
  df_readr <- read_csv("../data/sales_large.csv", show_col_types = FALSE)
})

In [ ]:
# Arrow read
system.time({
  df_arrow <- read_csv_arrow("../data/sales_large.csv")
})

In [ ]:
# Both give you a data frame you can use normally
head(df_arrow)

By default, `read_csv_arrow()` returns a regular R data frame. This is the easiest upgrade — just change the function name and your code gets faster.

But for even more power, you can keep the data in Arrow format.

## Arrow Tables

An **Arrow Table** is Arrow's version of a data frame. The data stays in Arrow's efficient columnar format instead of being converted to R's format.

To get an Arrow Table instead of a data frame, use `as_data_frame = FALSE` when reading:

In [ ]:
# Read as an Arrow Table (not an R data frame)
sales_arrow <- read_csv_arrow("../data/sales_large.csv", as_data_frame = FALSE)

class(sales_arrow)

In [ ]:
# Peek at the table
sales_arrow

In [ ]:
# Check dimensions
nrow(sales_arrow)
ncol(sales_arrow)

## Using dplyr on Arrow Tables

Just like sparklyr, you can use dplyr verbs on Arrow Tables. The operations are **lazy** — Arrow builds up a plan of what to do and only executes it when you ask for results with `collect()`.

This is efficient: if you filter 100K rows down to 50 and then collect, Arrow only materializes those 50 rows.

In [ ]:
# Filter and select (lazy — doesn't execute yet)
result <- sales_arrow |>
  filter(category == "Electronics") |>
  select(date, product, unit_price, quantity)

# This just shows the query plan
result

In [ ]:
# collect() executes the query and returns an R data frame
result |> collect() |> head(10)

In [ ]:
# Revenue by category (aggregation)
sales_arrow |>
  mutate(total_price = quantity * unit_price) |>
  group_by(category) |>
  summarize(
    total_revenue = sum(total_price),
    num_transactions = n()
  ) |>
  arrange(desc(total_revenue)) |>
  collect()

In [ ]:
# Revenue by region and category
sales_arrow |>
  mutate(total_price = quantity * unit_price) |>
  group_by(region, category) |>
  summarize(
    avg_price = mean(unit_price),
    total_qty = sum(quantity)
  ) |>
  arrange(region, desc(total_qty)) |>
  collect()

## open_dataset() — Larger-Than-Memory Workflows

For truly large data (files bigger than your RAM), Arrow's `open_dataset()` can work with files without loading them entirely. It reads data in chunks as needed.

This is especially powerful with multiple files. For example, if your data is split across many CSV or Parquet files in a folder, `open_dataset()` treats the entire folder as one dataset.

In [ ]:
# open_dataset works on files or directories
ds <- open_dataset("../data/sales_large.csv", format = "csv")

# Same dplyr syntax
ds |>
  filter(region == "West", category == "Electronics") |>
  summarize(avg_price = mean(unit_price)) |>
  collect()

## The Parquet File Format

CSV files are human-readable but inefficient for large data. **Parquet** is a modern columnar file format designed for analytics:

| Feature | CSV | Parquet |
|---------|-----|--------|
| Human-readable | Yes | No (binary) |
| File size | Large | Small (compressed) |
| Read speed | Slow | Very fast |
| Column selection | Must read whole file | Reads only needed columns |
| Data types | Everything is text | Types are preserved |

If you work with big data regularly, converting your CSVs to Parquet is one of the easiest wins.

In [ ]:
# Write our sales data as Parquet
write_parquet(sales_arrow, "../data/sales_large.parquet")

In [ ]:
# Compare file sizes
csv_size <- file.size("../data/sales_large.csv")
parquet_size <- file.size("../data/sales_large.parquet")

cat("CSV size:    ", round(csv_size / 1024 / 1024, 2), "MB\n")
cat("Parquet size:", round(parquet_size / 1024 / 1024, 2), "MB\n")
cat("Reduction:   ", round((1 - parquet_size / csv_size) * 100, 1), "%\n")

In [ ]:
# Reading Parquet is even faster than reading CSV with Arrow
system.time({
  df_parquet <- read_parquet("../data/sales_large.parquet")
})

In [ ]:
head(df_parquet)

## Clean Up

Let's remove the Parquet file we created (it was just for demonstration).

In [ ]:
file.remove("../data/sales_large.parquet")

## When to Use Arrow

Arrow is the right choice when:
- You want a simple speed boost for reading large CSV files (just swap `read_csv` for `read_csv_arrow`)
- You work with Parquet files
- Your data is large but fits on a single machine
- You want lazy evaluation (build a query, execute at the end)

Arrow is **not** the right choice when:
- Your data is distributed across many machines (use Spark)
- You need complex SQL queries (use DuckDB)
- Your data is small enough that base R / tidyverse works fine

---
## Summary

- **Arrow** stores data in an efficient columnar format that's faster to read and process
- `read_csv_arrow()` is a drop-in replacement for `read_csv()` — same result, faster
- Arrow Tables support dplyr verbs with lazy evaluation — `collect()` triggers execution
- **Parquet** is a compressed columnar file format that's much smaller and faster than CSV
- Arrow is lightweight (no Java/Spark needed) and great for single-machine big data work

**Next up:** [05 - Big Data with DuckDB](05-big-data-duckdb.ipynb) — fast SQL analytics directly on your files.